[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/04_data_preprocessing/01_missing_values_and_outliers.ipynb)

# Bài 01 — Xử lý missing value và outlier đúng cách

**🎯 Mục tiêu**
- Các chiến lược điền missing value (imputation) và khi nào dùng chiến lược nào.
- **Nguyên tắc vàng: chỉ `fit` trên train** — thấy tận mắt leakage khi vi phạm.
- Thêm cột chỉ báo missing; KNN/Iterative imputer.
- Xử lý outlier: clipping/winsorizing, biến đổi log, robust scaling — thay vì xoá.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/04_data_preprocessing

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.model_selection import train_test_split

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)
df = pd.read_csv("../../data/sample/house_prices_dirty.csv").drop_duplicates().reset_index(drop=True)
df["district"] = df["district"].str.strip().str.lower().str.replace("q.", "quận", regex=False).str.title()
df.isna().sum()

## 1. Các chiến lược xử lý missing

| Chiến lược | Ưu điểm | Nhược điểm | Khi nào dùng |
|---|---|---|---|
| Xoá hàng (`dropna`) | đơn giản | mất dữ liệu; gây **lệch** nếu không MCAR | thiếu rất ít và MCAR |
| Xoá cột | đơn giản | mất feature | cột thiếu quá nhiều (> 60–70%) và ít giá trị |
| Mean / median | nhanh | giảm phương sai, bỏ qua quan hệ giữa các biến | baseline; median khi có outlier |
| Mode / hằng số `"missing"` | cho biến phân loại | — | category |
| **+ cột chỉ báo** `is_missing` | giữ thông tin "bị thiếu" | thêm cột | khi việc thiếu có thể mang thông tin (MAR/MNAR) |
| KNN imputer | dùng quan hệ giữa các biến | chậm, cần scaling | dữ liệu vừa, quan hệ cục bộ |
| Iterative (model-based) | mạnh nhất trong nhóm đơn giản | chậm, phức tạp | khi missing quan trọng |
| Để model tự xử lý | không cần làm gì | chỉ một số model | HistGradientBoosting, XGBoost, LightGBM hỗ trợ NaN trực tiếp |

## 2. Nguyên tắc vàng: học thống kê từ TRAIN, áp dụng cho TEST

Imputer, scaler, encoder đều **học tham số từ dữ liệu** (median, mean, std, danh sách category...). Những tham số này phải học **chỉ từ tập train**. Nếu học từ toàn bộ dữ liệu, thông tin của tập test "rò rỉ" vào quá trình huấn luyện → điểm đánh giá lạc quan hơn thực tế.

In [ ]:
X = df[["area_m2", "bedrooms", "bathrooms", "age_years", "distance_to_center_km"]]
y = df["price_billion_vnd"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

imputer = SimpleImputer(strategy="median")
imputer.fit(X_train)                              # ✅ chỉ học median từ train
X_train_imp = imputer.transform(X_train)
X_test_imp = imputer.transform(X_test)            # áp dụng median CỦA TRAIN cho test

print("median học từ train:", dict(zip(X.columns, imputer.statistics_.round(2))))
print("còn NaN?", np.isnan(X_train_imp).any(), np.isnan(X_test_imp).any())

> Tại sao lại "khắt khe" vậy khi median của train và của toàn bộ dữ liệu gần như giống nhau? Vì (1) với các phép biến đổi phức tạp hơn (target encoding, feature selection, oversampling) leakage có thể làm điểm số sai lệch **rất nhiều** (chương 05); (2) khi triển khai, ta **không có** dữ liệu tương lai để tính thống kê — quy trình đánh giá phải mô phỏng đúng điều đó. Cách an toàn nhất: đặt mọi bước tiền xử lý trong một **Pipeline** (bài 04).

## 3. So sánh các chiến lược — cơ chế missing quyết định tất cả

Thí nghiệm: dữ liệu giá nhà đầy đủ (feature: log diện tích, số phòng, tuổi nhà, khoảng cách, quận, vườn). Ta **tự xoá** một phần giá trị `age_years` theo 2 cơ chế, rồi so sánh MAE (trên log giá) của Linear Regression với từng chiến lược. Model train trên dữ liệu **không thiếu** là mốc tốt nhất có thể ("oracle").

- **MCAR:** mỗi giá trị bị xoá với xác suất 30%, hoàn toàn ngẫu nhiên.
- **MNAR:** nhà càng **cũ** càng dễ bị thiếu tuổi (người bán không muốn ghi) — việc thiếu phụ thuộc chính giá trị bị thiếu.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_course.datasets import make_house_prices

full = make_house_prices(n=4000, seed=3)
num_cols = ["area_m2", "bedrooms", "bathrooms", "age_years", "distance_to_center_km"]
Xf = pd.concat([full[num_cols].astype(float),
                pd.get_dummies(full["district"], dtype=float, drop_first=True),
                full[["has_garden"]].astype(float)], axis=1)
Xf["area_m2"] = np.log(Xf["area_m2"])
yf = np.log(full["price_billion_vnd"])

mechanisms = {
    "MCAR": np.full(len(Xf), 0.3),
    "MNAR": np.clip((Xf["age_years"] - 5) / 30, 0, 1) * 0.9,
}
strategies = {
    "bỏ hẳn cột age_years": None,
    "mean": SimpleImputer(strategy="mean"),
    "median": SimpleImputer(strategy="median"),
    "median + cột chỉ báo": SimpleImputer(strategy="median", add_indicator=True),
    "KNN imputer (k=5)": KNNImputer(n_neighbors=5),
}
results = {}
for mech, p_missing in mechanisms.items():
    X = Xf.copy()
    X.loc[rng.random(len(X)) < p_missing, "age_years"] = np.nan
    Xtr, Xte, ytr, yte = train_test_split(X, yf, test_size=0.3, random_state=0)
    oracle = make_pipeline(StandardScaler(), LinearRegression()).fit(Xf.loc[Xtr.index], ytr)
    col = {"oracle (không thiếu)": mean_absolute_error(yte, oracle.predict(Xf.loc[Xte.index]))}
    for name, imp in strategies.items():
        if imp is None:
            m = make_pipeline(StandardScaler(), LinearRegression()).fit(Xtr.drop(columns="age_years"), ytr)
            col[name] = mean_absolute_error(yte, m.predict(Xte.drop(columns="age_years")))
        else:
            m = make_pipeline(StandardScaler(), imp, LinearRegression()).fit(Xtr, ytr)
            col[name] = mean_absolute_error(yte, m.predict(Xte))
    results[f"{mech} ({X['age_years'].isna().mean():.0%} thiếu)"] = col
pd.DataFrame(results).round(4)

**Đọc kết quả:**
- **Bỏ hẳn cột** luôn tệ nhất: mất toàn bộ thông tin của 70–75% giá trị còn lại.
- **MCAR:** mean/median gần như tốt bằng nhau và khá sát oracle; cột chỉ báo **không** giúp gì (việc thiếu chẳng mang thông tin).
- **MNAR:** điền median "kéo" những căn nhà cũ về tuổi trung bình → dự đoán giá quá cao cho nhà cũ. **Cột chỉ báo** cho model biết "căn này bị thiếu tuổi → nhiều khả năng là nhà cũ" và sửa được phần lớn sai số.
- **KNN imputer** không giúp ở đây vì tuổi nhà gần như không dự đoán được từ các feature khác. KNN/Iterative chỉ có lợi khi biến bị thiếu **tương quan** với các biến còn lại.

> Bài học: không có chiến lược "tốt nhất" chung — hãy hiểu **cơ chế missing** (chương 03, bài 04) và thử nghiệm bằng cross-validation. Median + cột chỉ báo là baseline an toàn.

## 4. Biến phân loại

In [ ]:
cats = pd.DataFrame({"payment": ["cash", np.nan, "card", "card", np.nan, "e-wallet"]})
print(SimpleImputer(strategy="most_frequent").fit_transform(cats).ravel())
print(SimpleImputer(strategy="constant", fill_value="missing").fit_transform(cats).ravel())

Với biến phân loại, điền `"missing"` như một **category riêng** thường tốt hơn điền mode: model có thể học được hành vi riêng của nhóm thiếu.

## 5. Xử lý outlier mà không xoá

In [ ]:
income = np.r_[rng.lognormal(np.log(15), 0.5, 995), [400, 520, 610, 800, 1000]]   # thu nhập (triệu/tháng) có vài giá trị cực lớn

lo, hi = np.percentile(income, [1, 99])
treatments = {
    "gốc": income,
    "clip 1%–99% (winsorize)": np.clip(income, lo, hi),
    "log1p": np.log1p(income),
    "robust scale (median/IQR)": (income - np.median(income)) / (np.percentile(income, 75) - np.percentile(income, 25)),
}
fig, axes = plt.subplots(1, 4, figsize=(18, 3.5))
for ax, (name, values) in zip(axes, treatments.items()):
    ax.hist(values, bins=50)
    ax.set_title(name)
plt.show()

| Kỹ thuật | Tác dụng | Lưu ý |
|---|---|---|
| **Clip/winsorize** theo phân vị | giới hạn ảnh hưởng của giá trị cực đoan | ngưỡng phân vị phải học từ **train** |
| **log / log1p** | nén đuôi phải, giảm lệch | chỉ cho giá trị dương (log1p cho ≥ 0) |
| **RobustScaler** | scale bằng median & IQR — outlier không làm méo thang đo của các điểm còn lại | outlier vẫn còn, chỉ không ảnh hưởng tới scaling |
| Model bền vững | tree-based, Huber loss, MAE loss | không cần xử lý outlier ở feature |
| **Xoá** | — | chỉ khi chắc chắn là **lỗi dữ liệu** |

Một transformer clip **học ngưỡng từ train** (kiểu dữ liệu sklearn — viết custom transformer kỹ hơn ở bài 04):

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin


class QuantileClipper(TransformerMixin, BaseEstimator):
    def __init__(self, lower=0.01, upper=0.99):
        self.lower, self.upper = lower, upper

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.low_ = np.nanquantile(X, self.lower, axis=0)
        self.high_ = np.nanquantile(X, self.upper, axis=0)
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.low_, self.high_)


clipper = QuantileClipper().fit(income.reshape(-1, 1))
print("ngưỡng học được:", clipper.low_.round(2), clipper.high_.round(2))
print("áp dụng cho dữ liệu mới:", clipper.transform(np.array([[5.0], [20.0], [3000.0]])).ravel())

## 🧠 Kiểm tra nhanh

1. Vì sao `SimpleImputer` phải `fit` trên train rồi mới `transform` test?
2. Khi nào nên thêm cột chỉ báo `is_missing`?
3. Vì sao "xoá hàng có missing" thường không phải lựa chọn tốt khi triển khai model?
4. Nêu 3 cách giảm ảnh hưởng của outlier mà không xoá dữ liệu.

## 📌 Tóm tắt

- Mọi transformer học tham số **chỉ từ train**; áp dụng cho validation/test/production.
- Median + cột chỉ báo là baseline mạnh; KNN/Iterative khi cần tốt hơn; một số model xử lý NaN trực tiếp.
- Biến phân loại: điền `"missing"` như một category.
- Outlier: clip theo phân vị (học từ train), log, robust scaling, model bền vững — xoá chỉ khi là lỗi.

## 📚 Tài liệu tham khảo

- scikit-learn User Guide — Imputation of missing values: https://scikit-learn.org/stable/modules/impute.html
- van Buuren — *Flexible Imputation of Missing Data* (miễn phí online)